# Python 与 LangChain 交互式学习笔记

这份 Notebook 是 `python_langchain_notes.md` 的配套运行版：Markdown 单元负责解释概念，Code 单元负责演示可以动手运行的例子。

使用建议：

- 先顺序阅读 Markdown，再运行后面的代码单元。
- Python 基础、Prompt、Runnable、Tool、Memory、简化 RAG 示例可以直接运行。
- DeepSeek 相关单元会检测 `DEEPSEEK_API_KEY`，没有配置时会跳过真实请求。
- 已补充 LangChain 官方常用例子：模型调用、工具调用、Agent、Memory、RAG、Runnable。
- Tavily、Chroma、真实 Embedding、外部 API 等示例默认作为可选扩展，不会强制运行。
- 当前项目使用 DeepSeek 模型，Notebook 会从项目根目录向上查找 `.env`；模型名优先读取 `DEEPSEEK_MODEL`，其次读取 `OPENAI_MODEL`，没有配置时使用 `deepseek-chat`。

In [2]:
# Notebook 环境说明：这里不自动安装依赖，只打印当前 Python 信息。
# 本项目使用 uv 管理依赖。需要新增依赖时，推荐在项目根目录执行：
# uv add langchain-tavily
# uv sync
#
# 注意：uv add 会更新 pyproject.toml / uv.lock，并同步项目环境。
# 如果你不想动项目 .venv，就需要把 Notebook kernel 切到已经安装该依赖的解释器。

import os
import sys
from pathlib import Path


def find_project_root(start: str | Path | None = None) -> Path:
    """从当前目录向上查找项目根目录，优先识别 .env / pyproject.toml。"""
    current = Path(start or Path.cwd()).resolve()
    candidates = [current, *current.parents]

    for path in candidates:
        if (path / ".env").exists() and (path / "pyproject.toml").exists():
            return path

    for path in candidates:
        if (path / ".env").exists():
            return path

    return current


PROJECT_ROOT = find_project_root()
ENV_PATH = PROJECT_ROOT / ".env"

print("Python:", sys.version)
print("当前解释器:", sys.executable)
print("当前目录:", Path.cwd())
print("项目根目录:", PROJECT_ROOT)
print(".env 路径:", ENV_PATH if ENV_PATH.exists() else "未找到 .env")
print("Notebook 已加载。")


Python: 3.14.2 (main, Jan 14 2026, 19:31:25) [MSC v.1944 64 bit (AMD64)]
当前解释器: D:\PythonProject\LearnOne\.venv\Scripts\python.exe
当前目录: D:\PythonProject\LearnOne\docs
项目根目录: D:\PythonProject\LearnOne
.env 路径: D:\PythonProject\LearnOne\.env
Notebook 已加载。


## 1. Python 基础语法

Python 的基础部分包括变量、数据类型、字符串、输入输出、类型转换、运算符和真假值判断。这些内容是后面写工具函数、处理 JSON、组织 Prompt 和解析模型结果的基础。

In [2]:
name = "Alice"
age = 18
price = 19.9
is_active = True
empty_value = None

print(type(name), name)
print(type(age), age)
print(type(price), price)
print(type(is_active), is_active)
print(type(empty_value), empty_value)

# 一次定义多个变量
x, y = 1, "Python"
print(x, y)

<class 'str'> Alice
<class 'int'> 18
<class 'float'> 19.9
<class 'bool'> True
<class 'NoneType'> None
1 Python


In [3]:
text = " Python,LangChain,RAG "

print(text.strip())
print(text.lower())
print(text.upper())
print(text.replace("RAG", "AI"))
print(text.split(","))
print("Python" in text)

name = "小明"
age = 20
print(f"姓名：{name}，年龄：{age}")

Python,LangChain,RAG
 python,langchain,rag 
 PYTHON,LANGCHAIN,RAG 
 Python,LangChain,AI 
[' Python', 'LangChain', 'RAG ']
True
姓名：小明，年龄：20


In [4]:
values = [False, None, 0, 0.0, "", [], {}, set(), (), "hello", [1]]

for value in values:
    print(repr(value), "=>", bool(value))

value = None
if value is None:
    print("判断 None 时推荐使用 is None")

False => False
None => False
0 => False
0.0 => False
'' => False
[] => False
{} => False
set() => False
() => False
'hello' => True
[1] => True
判断 None 时推荐使用 is None


## 2. 流程控制与容器

`if`、`for`、`while`、`match...case` 用来控制程序流程；`list`、`dict`、`tuple`、`set` 是最常用的数据容器。LangChain 应用里，消息列表、配置字典、文档集合、检索结果都离不开这些基础结构。

In [5]:
score = 88

if score >= 90:
    level = "优秀"
elif score >= 60:
    level = "及格"
else:
    level = "不及格"

print(level)

command = "start"
match command:
    case "start":
        print("启动")
    case "stop":
        print("停止")
    case _:
        print("未知命令")

及格
启动


In [6]:
names = ["Tom", "Jerry", "Alice"]

for index, name in enumerate(names, start=1):
    print(index, name)

person = {"name": "小明", "age": 18, "city": "上海"}
for key, value in person.items():
    print(key, value)

unique_tags = {"python", "langchain", "python", "rag"}
print(unique_tags)

1 Tom
2 Jerry
3 Alice
name 小明
age 18
city 上海
{'python', 'langchain', 'rag'}


## 3. 函数、类型注解与异常处理

工具调用和业务封装都依赖函数。类型注解能让代码更容易读，也能帮助 LangChain 推断工具参数。异常处理则让程序在外部请求、文件读取、模型调用失败时更稳。

In [7]:
def add(a: int, b: int) -> int:
    """计算两个整数之和。"""
    return a + b

print(add(3, 5))


def safe_divide(a: float, b: float) -> float | None:
    try:
        return a / b
    except ZeroDivisionError:
        print("除数不能为 0")
        return None

print(safe_divide(10, 2))
print(safe_divide(10, 0))

8
5.0
除数不能为 0
None


In [8]:
from dataclasses import dataclass


@dataclass
class Product:
    name: str
    price: float
    tags: list[str]


product = Product(name="保温杯", price=39.9, tags=["日用", "保温"])
print(product)
print(product.name, product.price)

Product(name='保温杯', price=39.9, tags=['日用', '保温'])
保温杯 39.9


## 4. 文件、环境变量与 JSON

大模型项目中，不要把 API Key 写死在代码里。推荐使用 `.env` 或系统环境变量。JSON 是模型 API、工具结果、结构化输出里最常见的数据交换格式。

In [9]:
import json
from pathlib import Path

sample = {
    "name": "LangChain",
    "type": "framework",
    "features": ["model", "prompt", "tool", "rag"],
}

text = json.dumps(sample, ensure_ascii=False, indent=2)
print(text)

loaded = json.loads(text)
print(loaded["features"])

# 演示 pathlib，不实际写业务文件
print(Path("docs") / "python_langchain_notes.md")

{
  "name": "LangChain",
  "type": "framework",
  "features": [
    "model",
    "prompt",
    "tool",
    "rag"
  ]
}
['model', 'prompt', 'tool', 'rag']
docs\python_langchain_notes.md


In [10]:
import os
from pathlib import Path


def find_project_root(start: str | Path | None = None) -> Path:
    """从当前目录向上查找项目根目录，优先识别 .env / pyproject.toml。"""
    current = Path(start or Path.cwd()).resolve()
    candidates = [current, *current.parents]

    for path in candidates:
        if (path / ".env").exists() and (path / "pyproject.toml").exists():
            return path

    for path in candidates:
        if (path / ".env").exists():
            return path

    return current


def load_project_env() -> Path | None:
    """加载项目根目录下的 .env；Notebook 位于 docs/ 时也能找到根目录配置。"""
    project_root = find_project_root()
    env_path = project_root / ".env"

    if not env_path.exists():
        print(f"未找到 .env，当前查找起点：{Path.cwd()}")
        return None

    try:
        from dotenv import load_dotenv
    except ImportError:
        print("未安装 python-dotenv，跳过 .env 加载。")
        return env_path

    load_dotenv(env_path, override=False)
    print(f"已加载 .env：{env_path}")
    return env_path


load_project_env()

print("DEEPSEEK_API_KEY 是否存在:", bool(os.getenv("DEEPSEEK_API_KEY")))
print("DEEPSEEK_MODEL:", os.getenv("DEEPSEEK_MODEL") or os.getenv("OPENAI_MODEL") or "deepseek-chat")

已加载 .env：D:\PythonProject\LearnOne\.env
DEEPSEEK_API_KEY 是否存在: True
DEEPSEEK_MODEL: deepseek-v4-flash


## 5. 大模型应用基础

大模型应用通常是传统程序和 LLM 的组合：

```text
用户输入 -> 传统应用预处理/权限/合规 -> 调用大模型 -> 解析结果 -> 业务处理 -> 返回
```

传统程序负责确定性控制，大模型负责自然语言理解、推理、生成、总结、分类和抽取。

In [11]:
def mock_llm_api(prompt: str) -> dict:
    """离线模拟大模型 API 返回。"""
    return {
        "model": "mock-deepseek",
        "content": f"我收到的问题是：{prompt}",
    }


def app_flow(user_input: str) -> str:
    cleaned = user_input.strip()
    if not cleaned:
        return "请输入有效问题。"
    response = mock_llm_api(cleaned)
    return response["content"]

print(app_flow("  什么是 LangChain？  "))

我收到的问题是：什么是 LangChain？


## 6. DeepSeek 模型初始化

本项目使用 DeepSeek。推荐优先使用 `langchain_deepseek.ChatDeepSeek`。

下面单元会检测 `DEEPSEEK_API_KEY`：

- 有 Key：初始化真实 DeepSeek 模型并发起一次简单调用。
- 没 Key：打印跳过提示，不报错。

In [1]:
import os
from pathlib import Path


def find_project_root(start: str | Path | None = None) -> Path:
    """从当前目录向上查找项目根目录，优先识别 .env / pyproject.toml。"""
    current = Path(start or Path.cwd()).resolve()
    candidates = [current, *current.parents]

    for path in candidates:
        if (path / ".env").exists() and (path / "pyproject.toml").exists():
            return path

    for path in candidates:
        if (path / ".env").exists():
            return path

    return current


def load_project_env() -> Path | None:
    """加载项目根目录下的 .env；Notebook 位于 docs/ 时也能找到根目录配置。"""
    project_root = find_project_root()
    env_path = project_root / ".env"

    if not env_path.exists():
        return None

    try:
        from dotenv import load_dotenv
    except ImportError:
        return env_path

    load_dotenv(env_path, override=False)
    return env_path


load_project_env()


def get_deepseek_model(temperature: float = 0):
    """创建 DeepSeek 聊天模型；没有 API Key 时返回 None。"""
    api_key = os.getenv("DEEPSEEK_API_KEY") or os.getenv("OPENAI_API_KEY")
    model_name = os.getenv("DEEPSEEK_MODEL") or os.getenv("OPENAI_MODEL") or "deepseek-chat"

    if not api_key:
        print("未配置 DEEPSEEK_API_KEY 或 OPENAI_API_KEY，跳过真实模型调用。")
        return None

    try:
        from langchain_deepseek import ChatDeepSeek
    except ImportError:
        print("未安装 langchain-deepseek，请先安装依赖。")
        return None

    return ChatDeepSeek(
        model=model_name,
        api_key=api_key,
        temperature=temperature,
    )


model = get_deepseek_model(temperature=0)
print("model:", type(model).__name__ if model else None)
print("model_name:", os.getenv("DEEPSEEK_MODEL") or os.getenv("OPENAI_MODEL") or "deepseek-chat")

model: ChatDeepSeek
model_name: deepseek-v4-flash


In [2]:
# 可选真实请求：有 DEEPSEEK_API_KEY 时运行。
if model:
    response = model.invoke("请用一句话解释 LangChain。")
    print(response.content)
else:
    print("没有真实模型对象，本单元只演示调用写法。")

LangChain是一个开源框架，用于通过模块化组件（如链、代理、记忆和工具集成）简化基于大语言模型（LLM）的应用程序开发与编排。


### 6.1 官方常用模型调用接口

LangChain 聊天模型常用接口包括：

- `invoke()`：单次调用。
- `batch()`：批量调用。
- `stream()`：流式输出。
- `with_retry()`：给 Runnable 增加重试策略。

下面单元只在 `model` 可用时真实请求 DeepSeek。没有模型时会跳过。

In [3]:
# 可选真实请求：DeepSeek invoke / batch / stream
if model:
    print("--- invoke ---")
    response = model.invoke("请用一句话解释 LCEL。")
    print(response.content)

    print("\n--- batch ---")
    batch_results = model.batch([
        "用一句话解释 Tool。",
        "用一句话解释 Agent。",
    ])
    for item in batch_results:
        print("-", item.content)

    print("\n--- stream ---")
    for chunk in model.stream("请用三句话介绍 RAG。"):
        print(chunk.content, end="")
else:
    print("未配置可用 DeepSeek 模型，跳过 invoke / batch / stream 示例。")

--- invoke ---
LCEL（LangChain Expression Language）是一种声明式语言，用于以简洁、可组合的方式定义和串联语言模型应用的组件（如提示、模型、输出解析器等）。

--- batch ---
- Tool 是一个可调用的功能模块，用于连接外部系统或执行特定操作，从而扩展AI智能体处理实际任务的能力。
- Agent是一个能够自主感知环境、作出决策并执行行动以实现特定目标的智能实体。

--- stream ---
RAG（检索增强生成）是一种结合信息检索与文本生成的技术，它先从一个知识库中检索与用户查询相关的文档片段。然后将这些检索到的内容作为上下文输入到大语言模型中，使其生成更准确、更具时效性的回答。这种方法有效解决了大模型知识更新滞后和幻觉问题，适用于问答系统、智能客服等需要外部知识的场景。

In [4]:
# Runnable 通用能力：with_retry。这里用离线函数演示，不触发真实请求。
from langchain_core.runnables import RunnableLambda

attempts = {"count": 0}


def sometimes_fail(text: str) -> str:
    attempts["count"] += 1
    if attempts["count"] == 1:
        raise ValueError("第一次故意失败，用于演示 retry")
    return f"成功处理：{text}"

retry_chain = RunnableLambda(sometimes_fail).with_retry(stop_after_attempt=2)
print(retry_chain.invoke("LangChain"))
print("调用次数:", attempts["count"])

成功处理：LangChain
调用次数: 2


## 7. LangChain 消息格式

LangChain 的聊天上下文由消息组成，常见类型包括：

- `SystemMessage`：系统角色、规则、背景。
- `HumanMessage`：用户输入。
- `AIMessage`：模型回复。
- `ToolMessage`：工具执行结果。

In [5]:
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage, ToolMessage

messages = [
    SystemMessage(content="你是一个严谨的 Python 老师。"),
    HumanMessage(content="解释一下 list 和 tuple 的区别。"),
    AIMessage(content="list 可变，tuple 不可变。"),
    ToolMessage(content="工具查询结果示例", tool_call_id="demo-tool-call"),
]

for message in messages:
    print(message.type, "=>", message.content)

system => 你是一个严谨的 Python 老师。
human => 解释一下 list 和 tuple 的区别。
ai => list 可变，tuple 不可变。
tool => 工具查询结果示例


In [6]:
# 字典形式也很常见，Agent 输入通常就是这种结构。
agent_input = {
    "messages": [
        {"role": "system", "content": "你是一个简洁的助手。"},
        {"role": "user", "content": "什么是 RAG？"},
    ]
}

print(agent_input)

{'messages': [{'role': 'system', 'content': '你是一个简洁的助手。'}, {'role': 'user', 'content': '什么是 RAG？'}]}


## 8. PromptTemplate 与 ChatPromptTemplate

Prompt 模板用于把变量填入提示词。普通文本用 `PromptTemplate`，聊天消息更常用 `ChatPromptTemplate`。

In [11]:
from langchain_core.prompts import PromptTemplate

prompt = PromptTemplate.from_template("请用{style}风格介绍{topic}")

print(prompt.format(style="通俗", topic="向量数据库"))
value = prompt.invoke({"style": "简洁通俗", "topic": "LangChain"})
print(value)

请用通俗风格介绍向量数据库
text='请用简洁通俗风格介绍LangChain'


In [13]:
from langchain_core.prompts import ChatPromptTemplate

chat_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "你是一个 Python 老师。"),
        ("human", "请解释：{question}"),
    ]
)

prompt_value = chat_prompt.invoke({"question": "什么是列表推导式？"})
print(prompt_value, 20*"-", type(prompt_value))

messages=[SystemMessage(content='你是一个 Python 老师。', additional_kwargs={}, response_metadata={}), HumanMessage(content='请解释：什么是列表推导式？', additional_kwargs={}, response_metadata={})] -------------------- <class 'langchain_core.prompt_values.ChatPromptValue'>


In [14]:
# 可选真实请求：Prompt -> DeepSeek
if model:
    response = model.invoke(chat_prompt.invoke({"question": "什么是列表推导式？"}))
    print(response.content)
else:
    print("未配置 DeepSeek，跳过真实模型调用。")

列表推导式（List Comprehension）是 Python 中一种简洁、高效的创建列表的方式。它用一行表达式替代传统的 `for` 循环 + `append()` 写法，让代码更紧凑、可读性更强。

### 基本结构
```python
[expression for item in iterable if condition]
```

- `expression`：对每个 `item` 执行的操作（可以是简单值、函数调用等）。
- `for item in iterable`：循环遍历可迭代对象（如列表、字符串、range）。
- `if condition`（可选）：过滤条件，只有满足条件的 `item` 才会被纳入列表。

### 示例

#### 1. 基本用法
把 0-9 的平方放入列表：
```python
# 传统方式
squares = []
for x in range(10):
    squares.append(x**2)

# 列表推导式
squares = [x**2 for x in range(10)]
```
结果：`[0, 1, 4, 9, 16, 25, 36, 49, 64, 81]`

#### 2. 加入条件
只保留偶数平方：
```python
even_squares = [x**2 for x in range(10) if x % 2 == 0]
# 输出：[0, 4, 16, 36, 64]
```

#### 3. 处理字符串
将字符串列表转换为大写：
```python
words = ['hello', 'world', 'python']
upper_words = [w.upper() for w in words]
# 输出：['HELLO', 'WORLD', 'PYTHON']
```

#### 4. 嵌套循环
扁平化一个二维列表：
```python
matrix = [[1, 2], [3, 4], [5, 6]]
flatten = [num for row in matrix for num in row]
# 输出：[1, 2, 3, 4, 5, 6]
```
注意：顺序与普通嵌套 `for` 循环一致。

### 对比传统循环
| 传统 `for` 循环 | 列表推导式 |

### 8.1 PromptValue 与消息转换

`ChatPromptTemplate.invoke()` 返回的是 `PromptValue`，它可以转成字符串，也可以转成消息列表。调试 Prompt 时很有用。

In [16]:
debug_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "你是一个{role}。"),
        ("human", "请解释：{topic}"),
    ]
)

prompt_value = debug_prompt.invoke({"role": "Python 老师", "topic": "生成器"})

print("--- to_string ---")
print(prompt_value.to_string(), type(prompt_value.to_string()))

print("\n--- to_messages ---")
for message in prompt_value.to_messages():
    print(message.type, message.content)

--- to_string ---
System: 你是一个Python 老师。
Human: 请解释：生成器 <class 'str'>

--- to_messages ---
system 你是一个Python 老师。
human 请解释：生成器


## 9. MessagesPlaceholder 与 Few-shot

`MessagesPlaceholder` 用来把历史消息插入 Prompt。Few-shot 用示例让模型模仿输出格式和判断逻辑。

In [17]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage

history_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "你是一个有帮助的助手。"),
        MessagesPlaceholder(variable_name="history"),
        ("human", "{question}"),
    ]
)

history_messages = [
    HumanMessage(content="我叫小明"),
    AIMessage(content="你好，小明。"),
]

value = history_prompt.invoke({"history": history_messages, "question": "我叫什么？"})
print(value)

messages=[SystemMessage(content='你是一个有帮助的助手。', additional_kwargs={}, response_metadata={}), HumanMessage(content='我叫小明', additional_kwargs={}, response_metadata={}), AIMessage(content='你好，小明。', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='我叫什么？', additional_kwargs={}, response_metadata={})]


In [ ]:
from langchain_core.prompts import FewShotPromptTemplate, PromptTemplate

example_prompt = PromptTemplate.from_template("输入：{input}\n输出：{output}")

few_shot_prompt = FewShotPromptTemplate(
    examples=[
        {"input": "质量很好", "output": "正面"},
        {"input": "物流太慢", "output": "负面"},
    ],
    example_prompt=example_prompt,
    suffix="输入：{text}\n输出：",
    input_variables=["text"],
)

print(few_shot_prompt.format(text="这件衣服穿起来很舒服"))

## 10. 输出解析与结构化输出

`StrOutputParser` 可以把模型消息转成字符串。结构化输出适合分类、抽取、表单填写等需要固定字段的场景。

In [22]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableLambda

# 不依赖真实模型的离线链：Prompt -> mock model -> parser
# print(prompt_value)
mock_model = RunnableLambda(lambda prompt_value: AIMessage(content=f"离线回答：{prompt_value.to_string()[:40]}..."))

chain = ChatPromptTemplate.from_template("请解释：{topic}") | mock_model | StrOutputParser()
print(chain.invoke({"topic": "RAG"}))

离线回答：Human: 请解释：RAG...


In [23]:
from pydantic import BaseModel, Field


class ProductInfo(BaseModel):
    name: str = Field(description="商品名称")
    price: float | None = Field(description="商品价格，如果没有则为 None")
    features: list[str] = Field(description="商品特点列表")


# 离线演示：直接构造 Pydantic 对象，理解结构化结果长什么样。
product = ProductInfo(name="商品A", price=199.0, features=["保暖", "适合秋冬"])
print(product)
print(product.model_dump())

name='商品A' price=199.0 features=['保暖', '适合秋冬']
{'name': '商品A', 'price': 199.0, 'features': ['保暖', '适合秋冬']}


In [24]:
# 可选真实请求：DeepSeek 结构化输出。
class CapitalInfo(BaseModel):
    name: str = Field(description="首都名称")
    location: str = Field(description="所在国家或地区")
    vibe: str = Field(description="城市气质或特点")
    economy: str = Field(description="经济特点")


if model:
    structured_model = model.with_structured_output(CapitalInfo)
    result = structured_model.invoke("请介绍一下法国首都巴黎。")
    print(result)
else:
    print("未配置 DeepSeek，跳过真实结构化输出调用。")

BadRequestError: Error code: 400 - {'error': {'message': 'Thinking mode does not support this tool_choice', 'type': 'invalid_request_error', 'param': None, 'code': 'invalid_request_error'}}

### 10.1 JsonOutputParser 示例

如果模型不支持原生结构化输出，可以用 `JsonOutputParser` 生成格式说明，并解析模型返回的 JSON 文本。

In [27]:
from langchain_core.output_parsers import JsonOutputParser
from langchain_core.prompts import PromptTemplate

json_parser = JsonOutputParser()

json_prompt = PromptTemplate.from_template(
    """请把下面主题整理成 JSON。\n{format_instructions}\n主题：{topic}"""
).partial(format_instructions=json_parser.get_format_instructions())

print(json_prompt.format(topic="LangChain 的核心模块"))

# 离线解析模型返回的 JSON 字符串
parsed = json_parser.invoke('{"name": "LangChain", "modules": ["models", "prompts", "tools", "rag"]}')
print(parsed)

请把下面主题整理成 JSON。
Return a JSON object.
主题：LangChain 的核心模块
{'name': 'LangChain', 'modules': ['models', 'prompts', 'tools', 'rag']}


## 11. LCEL、Runnable 与链式调用

LCEL 使用 `|` 把多个组件串起来。Runnable 统一了 `invoke`、`batch`、`stream`、`ainvoke` 等调用方式。

In [28]:
from langchain_core.runnables import RunnableLambda, RunnablePassthrough

strip_text = RunnableLambda(lambda x: x.strip())
to_upper = RunnableLambda(lambda x: x.upper())

simple_chain = strip_text | to_upper
print(simple_chain.invoke("  hello langchain  "))
print(simple_chain.batch([" a ", " b ", " c "]))

HELLO LANGCHAIN
['A', 'B', 'C']


In [29]:
def add_prefix(text: str) -> str:
    return f"结果：{text}"

chain = RunnableLambda(lambda x: x["question"]) | RunnableLambda(add_prefix)
print(chain.invoke({"question": "什么是 LCEL？"}))

passthrough_chain = RunnablePassthrough.assign(length=lambda x: len(x["text"]))
print(passthrough_chain.invoke({"text": "LangChain"}))

结果：什么是 LCEL？
{'text': 'LangChain', 'length': 9}


### 11.1 RunnableParallel 并行组织输入

官方示例里常见一种写法：用字典或 `RunnableParallel` 同时准备多个字段，再交给 Prompt。RAG 里经常用它把 `context` 和 `question` 合在一起。

In [30]:
from langchain_core.runnables import RunnableParallel

parallel = RunnableParallel(
    question=RunnableLambda(lambda x: x["question"]),
    uppercase=RunnableLambda(lambda x: x["question"].upper()),
    length=RunnableLambda(lambda x: len(x["question"])),
)

print(parallel.invoke({"question": "LangChain Runnable"}))

{'question': 'LangChain Runnable', 'uppercase': 'LANGCHAIN RUNNABLE', 'length': 18}


In [31]:
# pick 可以从 Runnable 输出字典中取指定字段。
pick_chain = parallel.pick("uppercase")
print(pick_chain.invoke({"question": "LangChain Runnable"}))

LANGCHAIN RUNNABLE


## 12. Tool 工具

Tool 是模型可以调用的外部函数。工具名、参数类型、docstring 和 Pydantic 字段说明都会影响模型调用质量。

In [ ]:
from langchain_core.tools import tool


@tool
def get_weather(city: str) -> str:
    """查询城市天气。city 应该是城市名称，例如 上海、北京。"""
    return f"{city} 今天晴，适合学习 LangChain。"


print(get_weather.name)
print(get_weather.description)
print(get_weather.invoke({"city": "上海"}))

In [ ]:
from typing import Literal
from pydantic import BaseModel, Field


class WeatherInput(BaseModel):
    location: str = Field(description="City name or coordinates")
    units: Literal["celsius", "fahrenheit"] = Field(
        default="celsius",
        description="Temperature unit preference",
    )
    include_forecast: bool = Field(
        default=False,
        description="Include 5-day forecast",
    )


@tool(args_schema=WeatherInput)
def get_weather_detail(
    location: str,
    units: str = "celsius",
    include_forecast: bool = False,
) -> str:
    """Get weather information for a location."""
    return f"{location} weather, units={units}, forecast={include_forecast}"


print(get_weather_detail.invoke({"location": "Shanghai", "units": "celsius", "include_forecast": True}))

### 12.1 bind_tools 与 tool_calls

官方工具调用流程通常是：

```text
model.bind_tools(tools) -> AIMessage.tool_calls -> 执行工具 -> ToolMessage -> 再交给模型总结
```

`bind_tools` 让模型知道有哪些工具可用；`AIMessage.tool_calls` 里会包含模型想调用的工具名称和参数。

In [ ]:
# 可选真实请求：DeepSeek bind_tools -> tool_calls
if model:
    model_with_tools = model.bind_tools([get_weather])
    ai_message = model_with_tools.invoke("上海天气怎么样？")

    print("content:", ai_message.content)
    print("tool_calls:", ai_message.tool_calls)
else:
    print("未配置可用 DeepSeek 模型，跳过 bind_tools 真实调用。")

In [ ]:
# 离线演示：手动构造 tool_calls，并把工具结果包装成 ToolMessage。
from langchain_core.messages import AIMessage, ToolMessage

fake_ai_message = AIMessage(
    content="",
    tool_calls=[
        {
            "name": "get_weather",
            "args": {"city": "上海"},
            "id": "call-weather-1",
            "type": "tool_call",
        }
    ],
)

available_tools = {get_weather.name: get_weather}

tool_messages = []
for tool_call in fake_ai_message.tool_calls:
    selected_tool = available_tools[tool_call["name"]]
    tool_result = selected_tool.invoke(tool_call["args"])
    tool_messages.append(
        ToolMessage(content=tool_result, tool_call_id=tool_call["id"])
    )

for message in tool_messages:
    print(message.type, message.content, message.tool_call_id)

### 12.2 直接创建结构化工具

除了 `@tool`，也可以用 `StructuredTool.from_function()` 把普通函数转成工具。这适合你想显式控制工具名、描述和参数 schema 的时候。

In [ ]:
from langchain_core.tools import StructuredTool


def multiply(a: int, b: int) -> int:
    """Multiply two integers."""
    return a * b

multiply_tool = StructuredTool.from_function(
    func=multiply,
    name="multiply",
    description="Multiply two integers together.",
)

print(multiply_tool.name)
print(multiply_tool.invoke({"a": 6, "b": 7}))

## 13. Agent 智能体

Agent 可以让模型根据任务自行选择工具、执行工具、观察结果并继续推理。简单理解：

```text
Agent = 大语言模型 + 工具集 + 决策逻辑 + 记忆/状态
```

下面真实 Agent 单元只有配置 DeepSeek 后才会运行。

In [ ]:
# 可选真实请求：DeepSeek + Tool + Agent
if model:
    from langchain.agents import create_agent

    agent = create_agent(
        model=model,
        tools=[get_weather],
        system_prompt="你是一个可以调用工具的助手。如果用户问天气，必须调用天气工具。",
    )

    result = agent.invoke(
        {"messages": [{"role": "user", "content": "上海天气怎么样？"}]}
    )
    print(result["messages"][-1].content)
else:
    print("未配置 DeepSeek，跳过真实 Agent 调用。")

In [ ]:
# Agent 输入输出结构示例，不依赖真实模型。
messages = [
    {"role": "user", "content": "我叫小明"},
    {"role": "assistant", "content": "你好，小明。"},
    {"role": "user", "content": "我叫什么？"},
]

agent_state = {"messages": messages}
print(agent_state)
print("最后一条消息:", agent_state["messages"][-1])

### 13.1 Agent 结构化输出 response_format

官方 Agent 支持 `response_format`，可以让 Agent 最终返回结构化结果。返回对象通常在 `structured_response` 字段里。

In [ ]:
from pydantic import BaseModel, Field


class WeatherAnswer(BaseModel):
    city: str = Field(description="城市")
    weather: str = Field(description="天气结论")
    used_tool: bool = Field(description="是否使用了工具")


if model:
    from langchain.agents import create_agent

    structured_agent = create_agent(
        model=model,
        tools=[get_weather],
        system_prompt="你是天气助手。回答天气问题时先调用工具。",
        response_format=WeatherAnswer,
    )

    result = structured_agent.invoke(
        {"messages": [{"role": "user", "content": "上海天气怎么样？"}]}
    )
    print(result.get("structured_response"))
else:
    print("未配置可用 DeepSeek 模型，跳过 Agent response_format 示例。")

### 13.2 Agent 流式输出

Agent 可以用 `stream()` 查看中间消息。调试工具调用时，流式输出比只看最终答案更清楚。

In [ ]:
if model:
    from langchain.agents import create_agent

    stream_agent = create_agent(
        model=model,
        tools=[get_weather],
        system_prompt="你是一个会调用工具的助手。",
    )

    for chunk in stream_agent.stream(
        {"messages": [{"role": "user", "content": "上海天气怎么样？"}]},
        stream_mode="values",
    ):
        latest = chunk["messages"][-1]
        print(latest.type, latest.content)
else:
    print("未配置可用 DeepSeek 模型，跳过 Agent stream 示例。")

## 14. Tavily 网页搜索工具

Tavily 是常见联网搜索工具，需要 `TAVILY_API_KEY` 和 `langchain-tavily` 依赖。

本项目使用 `uv` 管理依赖，不需要非得用 `pip`。推荐在项目根目录执行：

```powershell
uv add langchain-tavily
uv sync
```

需要注意：

- `uv add langchain-tavily` 会把依赖写入 `pyproject.toml` 和 `uv.lock`。
- `uv sync` 会同步项目环境，通常也就是当前项目 `.venv`。
- 如果你不想动项目 `.venv`，就不能通过 `uv add/sync` 把包装进这个 kernel；这时应该把 Notebook kernel 切换到已经安装了 `langchain-tavily` 的解释器。

这个单元可以单独运行；即使你没有先运行前面的环境初始化单元，它也会自己查找项目根目录并加载 `.env`。


In [7]:
# 可选：使用 uv 安装 Tavily 依赖。
# 运行这个单元会修改 pyproject.toml / uv.lock，并同步项目环境。
# 如果你不想动项目 .venv，请不要运行这个单元，改为切换 Notebook kernel。

import subprocess
from pathlib import Path

project_root = find_project_root() if "find_project_root" in globals() else Path.cwd().resolve().parent
print("项目根目录:", project_root)

# 取消下面两行注释后再运行：
# subprocess.run(["uv", "add", "langchain-tavily"], cwd=project_root, check=True)
# subprocess.run(["uv", "sync"], cwd=project_root, check=True)


项目根目录: D:\PythonProject\LearnOne


In [9]:
import importlib.util
import os
import sys

from pathlib import Path


def find_project_root(start: str | Path | None = None) -> Path:
    """从当前目录向上查找项目根目录，优先识别 .env / pyproject.toml。"""
    current = Path(start or Path.cwd()).resolve()
    candidates = [current, *current.parents]

    for path in candidates:
        if (path / ".env").exists() and (path / "pyproject.toml").exists():
            return path

    for path in candidates:
        if (path / ".env").exists():
            return path

    return current


def load_project_env() -> Path | None:
    """加载项目根目录下的 .env；Notebook 位于 docs/ 时也能找到根目录配置。"""
    project_root = find_project_root()
    env_path = project_root / ".env"

    if not env_path.exists():
        print(f"未找到 .env，当前查找起点：{Path.cwd()}")
        return None

    try:
        from dotenv import load_dotenv
    except ImportError:
        print("未安装 python-dotenv，跳过 .env 加载。")
        return env_path

    load_dotenv(env_path, override=False)
    print(f"已加载 .env：{env_path}")
    return env_path

load_project_env()

print("当前解释器:", sys.executable)
print("TAVILY_API_KEY 是否存在:", bool(os.getenv("TAVILY_API_KEY")))
print("langchain_tavily 是否可导入:", importlib.util.find_spec("langchain_tavily") is not None)

if not os.getenv("TAVILY_API_KEY"):
    print("未配置 TAVILY_API_KEY，跳过 Tavily 搜索。")
elif importlib.util.find_spec("langchain_tavily") is None:
    print("当前 kernel 没有安装 langchain-tavily。")
    print("如果要装到项目环境，请在项目根目录运行：uv add langchain-tavily && uv sync")
    print("如果不想动 .venv，请切换到已经安装 Tavily 的 Notebook kernel。")
else:
    from langchain_tavily import TavilySearch

    search_tool = TavilySearch(max_results=3, topic="general")
    result = search_tool.invoke({"query": "查看下今日北京的天气"})
    print(result)


已加载 .env：D:\PythonProject\LearnOne\.env
当前解释器: D:\PythonProject\LearnOne\.venv\Scripts\python.exe
TAVILY_API_KEY 是否存在: True
langchain_tavily 是否可导入: True
{'query': '查看下今日北京的天气', 'follow_up_questions': None, 'answer': None, 'images': [], 'results': [{'url': 'https://www.ventusky.com/zh-tw/beijing', 'title': '天氣- 北京市- 14天預報：氣溫、風和雷達 - Ventusky', 'content': '# [Ventusky](https://www.ventusky.com/zh-tw). [](https://www.ventusky.com/zh-tw/beijing "找出我的位置")[](https://www.ventusky.com/zh-tw/beijing)*   [](https://www.ventusky.com/zh-tw/beijing). *   [](https://www.ventusky.com/zh-tw/beijing). *   [](https://www.ventusky.com/zh-tw/beijing). *   [](https://www.ventusky.com/zh-tw/beijing). *   [預測](https://www.ventusky.com/zh-tw/forecast/). [](https://www.ventusky.com/zh-tw/beijing#). [世界](https://www.ventusky.com/zh-tw/forecast) / [中国](https://www.ventusky.com/zh-tw/forecast/china) / [北京|北京](https://www.ventusky.com/zh-tw/forecast/china/beijing-shi) (Beijing Shi). *   [當前天氣](https://www.ventusky.

## 15. 记忆与聊天历史

短期记忆保存当前会话上下文，长期记忆跨会话持久化。`RunnableWithMessageHistory` 可以给链增加历史对话能力。

In [ ]:
from langchain_core.chat_history import InMemoryChatMessageHistory

history = InMemoryChatMessageHistory()
history.add_user_message("你好")
history.add_ai_message("你好，有什么可以帮你？")

for message in history.messages:
    print(message.type, message.content)

In [ ]:
from langchain_core.runnables.history import RunnableWithMessageHistory

store = {}


def get_session_history(session_id: str):
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]


# 离线模拟带历史的链：把 question 转成 AIMessage
history_demo_chain = RunnableLambda(lambda x: AIMessage(content=f"收到：{x['question']}"))

chain_with_history = RunnableWithMessageHistory(
    history_demo_chain,
    get_session_history,
    input_messages_key="question",
)

result1 = chain_with_history.invoke(
    {"question": "我叫小明"},
    config={"configurable": {"session_id": "user-1"}},
)
result2 = chain_with_history.invoke(
    {"question": "我叫什么？"},
    config={"configurable": {"session_id": "user-1"}},
)

print(result1.content)
print(result2.content)
print("历史条数:", len(store["user-1"].messages))

### 15.1 Agent 短期记忆：checkpointer + thread_id

官方 Agent 的短期记忆通常通过 checkpointer 保存，同一个 `thread_id` 会复用历史上下文。这个例子需要 `langgraph`，并且只有 `model` 可用时才真实调用。

In [ ]:
if model:
    try:
        from langchain.agents import create_agent
        from langgraph.checkpoint.memory import InMemorySaver

        checkpointer = InMemorySaver()
        memory_agent = create_agent(
            model=model,
            tools=[],
            checkpointer=checkpointer,
            system_prompt="你是一个能记住当前会话上下文的助手。",
        )

        config = {"configurable": {"thread_id": "demo-user-1"}}

        memory_agent.invoke(
            {"messages": [{"role": "user", "content": "我叫小明"}]},
            config=config,
        )
        result = memory_agent.invoke(
            {"messages": [{"role": "user", "content": "我叫什么？"}]},
            config=config,
        )
        print(result["messages"][-1].content)
    except ImportError:
        print("未安装 langgraph，跳过 Agent checkpointer 示例。")
else:
    print("未配置可用 DeepSeek 模型，跳过 Agent 短期记忆示例。")

## 16. RAG 核心概念

RAG 是 Retrieval-Augmented Generation：先检索相关资料，再把资料放进 Prompt，让模型基于资料回答。

```text
用户问题 -> 检索相关资料 -> 组装 Prompt -> 模型生成答案
```

下面先用纯 Python 做一个离线版“小 RAG”，理解流程。

In [ ]:
from dataclasses import dataclass


@dataclass
class SimpleDocument:
    page_content: str
    metadata: dict


docs = [
    SimpleDocument("LangChain 用于构建大模型应用，包含 Prompt、Tool、Agent、RAG 等模块。", {"source": "note-1"}),
    SimpleDocument("RAG 会先检索知识库，再让模型根据检索结果回答。", {"source": "note-2"}),
    SimpleDocument("Python 的 pathlib 适合处理文件路径。", {"source": "note-3"}),
]


def simple_retrieve(query: str, documents: list[SimpleDocument], k: int = 2) -> list[SimpleDocument]:
    keywords = set(query.lower().split())

    def score(doc: SimpleDocument) -> int:
        text = doc.page_content.lower()
        return sum(1 for keyword in keywords if keyword in text)

    return sorted(documents, key=score, reverse=True)[:k]


question = "LangChain RAG 是什么"
retrieved = simple_retrieve(question, docs)
for doc in retrieved:
    print(doc.metadata["source"], doc.page_content)

In [ ]:
def format_docs(documents: list[SimpleDocument]) -> str:
    return "\n".join(f"- {doc.page_content}" for doc in documents)

rag_prompt = ChatPromptTemplate.from_template(
    """请只根据资料回答问题。

资料：
{context}

问题：{question}
"""
)

context = format_docs(retrieved)
prompt_value = rag_prompt.invoke({"context": context, "question": question})
print(prompt_value.to_string())

In [ ]:
# 可选真实请求：把离线检索结果交给 DeepSeek 回答。
if model:
    answer = model.invoke(prompt_value)
    print(answer.content)
else:
    print("未配置 DeepSeek，跳过真实 RAG 生成。")

## 17. 文档加载与文本切分

LangChain 的 `Document` 对象包含 `page_content` 和 `metadata`。长文本通常需要切分后再进入向量库。

In [ ]:
from langchain_core.documents import Document

langchain_doc = Document(
    page_content="LangChain 是一个用于构建大模型应用的框架。",
    metadata={"source": "manual", "chapter": "intro"},
)

print(langchain_doc.page_content)
print(langchain_doc.metadata)

In [ ]:
try:
    from langchain_text_splitters import RecursiveCharacterTextSplitter

    long_text = "LangChain 可以处理模型、提示词、工具、Agent、记忆和 RAG。" * 5
    splitter = RecursiveCharacterTextSplitter(chunk_size=40, chunk_overlap=8)
    chunks = splitter.split_text(long_text)
    for index, chunk in enumerate(chunks, start=1):
        print(index, chunk)
except ImportError:
    print("未安装 langchain-text-splitters，跳过文本切分示例。")

## 18. Embedding、向量存储与 Retriever

真实向量检索需要 Embedding 模型和向量库。为了让 Notebook 可以离线运行，这里先用简单关键词检索理解 Retriever 的接口思想。真实项目可替换为 Chroma、FAISS、Milvus 等。

In [ ]:
class SimpleRetriever:
    def __init__(self, documents: list[SimpleDocument]):
        self.documents = documents

    def invoke(self, query: str) -> list[SimpleDocument]:
        return simple_retrieve(query, self.documents, k=2)


retriever = SimpleRetriever(docs)
results = retriever.invoke("RAG 知识库")
for doc in results:
    print(doc.metadata, doc.page_content)

In [ ]:
# 可选：Chroma 写法示意。这里不自动创建向量库，避免额外耗时和外部依赖。
print("真实项目常见流程：")
print("1. loader 加载文档")
print("2. splitter 切分文档")
print("3. embeddings 生成向量")
print("4. vector_store.add_documents 写入向量库")
print("5. retriever.invoke(query) 检索")

### 18.1 InMemoryVectorStore 标准接口

官方 RAG 教程常用 VectorStore 的标准接口：

- `add_documents()`：写入文档。
- `similarity_search()`：相似度搜索。
- `as_retriever()`：转成 Retriever。

下面用一个离线 `KeywordEmbeddings` 模拟 Embedding，避免依赖外部向量模型。

In [ ]:
from langchain_core.embeddings import Embeddings
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_core.documents import Document


class KeywordEmbeddings(Embeddings):
    vocabulary = ["langchain", "rag", "agent", "tool", "prompt", "python"]

    def embed_query(self, text: str) -> list[float]:
        lowered = text.lower()
        return [float(lowered.count(word)) for word in self.vocabulary]

    def embed_documents(self, texts: list[str]) -> list[list[float]]:
        return [self.embed_query(text) for text in texts]


vector_docs = [
    Document(page_content="LangChain 支持模型、Prompt、Tool 和 Agent。", metadata={"source": "lc"}),
    Document(page_content="RAG 通过检索增强大模型回答。", metadata={"source": "rag"}),
    Document(page_content="Python 函数可以封装成 LangChain Tool。", metadata={"source": "tool"}),
]

vector_store = InMemoryVectorStore(KeywordEmbeddings())
vector_store.add_documents(vector_docs)

results = vector_store.similarity_search("LangChain RAG", k=2)
for doc in results:
    print(doc.metadata, doc.page_content)

In [ ]:
retriever = vector_store.as_retriever(search_kwargs={"k": 2})
retrieved_docs = retriever.invoke("怎么用工具和 RAG？")

for doc in retrieved_docs:
    print(doc.metadata["source"], doc.page_content)

### 18.2 官方 RAG 链写法：Retriever -> format_docs -> Prompt -> Model

真实项目里，常见 RAG 链会把检索器、格式化函数、Prompt、模型和输出解析器串起来。

In [ ]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough


def format_langchain_docs(documents: list[Document]) -> str:
    return "\n\n".join(doc.page_content for doc in documents)

rag_chain_offline = (
    {
        "context": retriever | RunnableLambda(format_langchain_docs),
        "question": RunnablePassthrough(),
    }
    | ChatPromptTemplate.from_template(
        """请只根据资料回答问题。\n\n资料：\n{context}\n\n问题：{question}"""
    )
    | mock_model
    | StrOutputParser()
)

print(rag_chain_offline.invoke("LangChain 支持哪些能力？"))

## 19. Streamlit + LangChain 常见模式

Streamlit 做聊天应用时，通常用 `st.session_state` 保存历史消息，用 LangChain 负责模型、RAG 或 Agent。Notebook 中不启动 Streamlit，只保留核心状态思路。

In [ ]:
# Streamlit session_state 的思想可以用普通字典理解。
session_state = {}

if "messages" not in session_state:
    session_state["messages"] = []

session_state["messages"].append({"role": "user", "content": "你好"})
session_state["messages"].append({"role": "assistant", "content": "你好，有什么可以帮你？"})

print(session_state["messages"])

## 20. 常见错误排查

常见问题：

- 环境变量没加载：检查 `.env` 路径、变量名、Notebook 工作目录。
- 工具不被调用：检查工具名、docstring、参数类型、system prompt。
- Agent 编造结果：要求必须基于工具或检索结果回答，没有资料就说不知道。
- 多轮对话丢上下文：检查是否把历史消息重新传入模型或 Agent。
- RAG 答案不准：检查切分、检索参数、Prompt 约束、来源返回。
- 导入路径报错：LangChain 版本变化快，以当前依赖和官方文档为准。

In [ ]:
import importlib.util
import os
import sys

from pathlib import Path


def find_project_root(start: str | Path | None = None) -> Path:
    """从当前目录向上查找项目根目录，优先识别 .env / pyproject.toml。"""
    current = Path(start or Path.cwd()).resolve()
    candidates = [current, *current.parents]

    for path in candidates:
        if (path / ".env").exists() and (path / "pyproject.toml").exists():
            return path

    for path in candidates:
        if (path / ".env").exists():
            return path

    return current


def load_project_env() -> Path | None:
    """加载项目根目录下的 .env；Notebook 位于 docs/ 时也能找到根目录配置。"""
    project_root = find_project_root()
    env_path = project_root / ".env"

    if not env_path.exists():
        print(f"未找到 .env，当前查找起点：{Path.cwd()}")
        return None

    try:
        from dotenv import load_dotenv
    except ImportError:
        print("未安装 python-dotenv，跳过 .env 加载。")
        return env_path

    load_dotenv(env_path, override=False)
    print(f"已加载 .env：{env_path}")
    return env_path

load_project_env()

checks = {
    "当前解释器": sys.executable,
    "DEEPSEEK_API_KEY": bool(os.getenv("DEEPSEEK_API_KEY")),
    "OPENAI_API_KEY": bool(os.getenv("OPENAI_API_KEY")),
    "DEEPSEEK_MODEL": os.getenv("DEEPSEEK_MODEL") or os.getenv("OPENAI_MODEL") or "deepseek-chat",
    "TAVILY_API_KEY": bool(os.getenv("TAVILY_API_KEY")),
    "langchain_tavily 可导入": importlib.util.find_spec("langchain_tavily") is not None,
    "langchain_deepseek 可导入": importlib.util.find_spec("langchain_deepseek") is not None,
}

for key, value in checks.items():
    print(f"{key}: {value}")


### 20.1 本 Notebook 对照的官方主题

补充示例主要参考这些官方主题：

- Chat models：`invoke`、`batch`、`stream`。
- Prompt templates：`ChatPromptTemplate`、`PromptValue`。
- Tools：`@tool`、`bind_tools`、`tool_calls`、`ToolMessage`。
- Agents：`create_agent`、`response_format`、streaming、short-term memory。
- RAG：`Document`、Text Splitter、VectorStore、Retriever、RAG Chain。
- Runnable：LCEL、`RunnableLambda`、`RunnableParallel`、`with_retry`。

## 22. 官方参考链接

- LangChain Overview: https://docs.langchain.com/oss/python/langchain/overview
- Chat models: https://docs.langchain.com/oss/python/langchain-models
- Prompt templates: https://docs.langchain.com/oss/python/langchain/prompts
- Tools: https://docs.langchain.com/oss/python/langchain/tools
- Agents: https://docs.langchain.com/oss/python/langchain/agents
- Short-term memory: https://docs.langchain.com/oss/python/langchain/short-term-memory
- RAG: https://docs.langchain.com/oss/python/langchain/rag
- Vector stores: https://docs.langchain.com/oss/python/integrations/vectorstores/
- DeepSeek integration: https://docs.langchain.com/oss/python/integrations/chat/deepseek
